### 1. Load the data

In [1]:
from pathlib import Path
import json

import pandas as pd


RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

nodes = pd.read_csv(
    RAW_DATA_DIR / "isebel-mecklenburg-nodes.csv"
)

edges = pd.read_csv(
    RAW_DATA_DIR / "isebel-mecklenburg-edges.csv"
)

community_assignments = pd.read_csv(
    PROCESSED_DATA_DIR / "community_assignments.csv"
)

community_assignments["story_id"] = (
    community_assignments["story_id"].astype(str)
)

### 2. Extract place names

In [2]:
def get_place_name(properties):
    properties = json.loads(properties)
    return str(properties.get("name", "")).strip()


places = nodes[nodes["label"] == "place"].copy()

places["place_name"] = places["properties"].apply(
    get_place_name
)

place_names = places.set_index("id")["place_name"]

### 3. Extract story-place relationships

In [3]:
story_places = (
    edges.loc[
        edges["label"] == "place-of-narration",
        ["src-id", "dst-id"]
    ]
    .rename(columns={
        "src-id": "story_id",
        "dst-id": "place_id"
    })
    .drop_duplicates()
)

story_places["story_id"] = (
    story_places["story_id"].astype(str)
)

story_places["place_name"] = (
    story_places["place_id"].map(place_names)
)

display(story_places.head())

,story_id,place_id,place_name
4644,13978,2275,Bartelshagen
4645,12426,176,Wustrow
4646,10185,5221,Gresenhorst
4647,8428,2294,Tramm
4648,15151,7392,Zarrentin


### 4. Join places with Louvain communities

In [4]:
community_places = story_places.merge(
    community_assignments[
        ["story_id", "louvain_community"]
    ],
    on="story_id",
    how="inner"
)

print("Joined story-place rows:", len(community_places))

Joined story-place rows: 1042


### 5. Count places in each community

In [5]:
place_counts = (
    community_places
    .groupby(
        ["louvain_community", "place_name"]
    )["story_id"]
    .nunique()
    .reset_index(name="story_count")
)

place_counts = place_counts.sort_values(
    [
        "louvain_community",
        "story_count",
        "place_name"
    ],
    ascending=[
        True,
        False,
        True
    ]
)

### 6. Show the top places

In [6]:
TOP_PLACES = 10

top_places = (
    place_counts
    .groupby("louvain_community")
    .head(TOP_PLACES)
    .reset_index(drop=True)
)

for community_id, group in top_places.groupby(
    "louvain_community"
):
    print(f"\nCommunity {community_id}")

    for _, row in group.iterrows():
        print(
            f"- {row['place_name']}: "
            f"{row['story_count']} stories"
        )


Community 0
- Gresenhorst: 3 stories
- Teterow: 2 stories
- Bad Doberan: 1 stories
- Biendorf: 1 stories
- Dargun: 1 stories
- Grevesmühlen: 1 stories
- Langen Trechow: 1 stories
- Parchim: 1 stories

Community 1
- Althagen: 5 stories
- Eldena: 4 stories
- Ribnitz-Damgarten: 4 stories
- Waren: 4 stories
- Bad Doberan: 3 stories
- Feldberg: 3 stories
- Kirchdorf auf Poel: 3 stories
- Parchim: 3 stories
- Schönberg: 3 stories
- Fürstenhagen: 2 stories

Community 2
- Ribnitz-Damgarten: 10 stories
- Kühlungsborn West: 2 stories
- Tessin: 2 stories
- Warnemünde: 2 stories
- Wismar: 2 stories
- Althagen: 1 stories
- Bartelshagen: 1 stories
- Damgartener Gegend: 1 stories
- Eldena: 1 stories
- Gelbensande: 1 stories

Community 3
- Feldberg: 6 stories
- Neustrelitz: 4 stories
- Malchow: 3 stories
- Neukloster: 2 stories
- Sülze: 2 stories
- Waren: 2 stories
- Wismar: 2 stories
- Barkow: 1 stories
- Bartenshagen: 1 stories
- Basedow: 1 stories

Community 4
- Schönberg: 2 stories
- Boitin-Resdo

### 7. Save the result

In [7]:
top_places.to_csv(
    PROCESSED_DATA_DIR / "community_top_places.csv",
    index=False
)